In [ ]:
from reaction_lammps_mupt.initialization import initialize
from reaction_lammps_mupt.input_parser import InputParser
from reaction_lammps_mupt.cache import GetCacheDir, RunDirectoryManager, RetentionCleanup
from reaction_lammps_mupt.detectors.detector import Detector
from reaction_lammps_mupt.reaction_template_builder.run_reaction_template_pipeline import ReactionTemplatePipeline
import json

initialize()
input_parser = InputParser()
cache_dir = GetCacheDir().staging_dir
dated_cache_dir = RunDirectoryManager.make_dated_run_dir(cache_dir, chdir_to="none")
# #future use
# RunDirectoryManager.copy_into_run(cache_dir, dated_cache_dir)
RetentionCleanup.run(GetCacheDir().cache_base_dir)

In [ ]:
import importlib

m = importlib.import_module(
    "reaction_lammps_mupt.reaction_template_builder.run_reaction_template_pipeline"
)
print("Module loaded from:", m.__file__)
print("Has RunReactionTemplatePipeline?", hasattr(m, "RunReactionTemplatePipeline"))
print("Top-level names (filtered):")
print([n for n in dir(m) if "Pipeline" in n or "pipeline" in n or "Reaction" in n])


In [ ]:
# or user can use a json file to read inputs
with open("inputs.json", "r") as f:
    inputs = json.load(f)
validated_inputs = input_parser.validate_inputs(inputs)   # validates again, returns dict


In [ ]:
# Example of using retention_cleanup
# This function will delete files in the cache directory that are older than the 
# specified number of days (e.g., 30 days)
# retention_cleanup(default_cache_dir, retention_days=30)

In [ ]:
# Detector Has to called Now after the inputs are validated, because it needs the input dict to run the detection workflow
detector = Detector(inputs)
detected_reactions = detector.reactions_dict
non_monomers = detector.non_reactants_list

In [ ]:
%%capture captured_output
reaction_template_pipeline = ReactionTemplatePipeline(validated_inputs, detected_reactions, non_monomers)
reaction_template_pipeline.run()


In [ ]:
img = reaction_template_pipeline.molecule_images
display(img)

In [ ]:
%%capture captured_output
execute_pipeline(
        detected_reactions=detected_reactions,
        retain_smiles=non_monomers,
        cache=cache_dir,
    )

In [ ]:

from pathlib import Path
with open(Path(cache_dir) / "output.txt", "w") as f:
    f.write(captured_output.stdout)
print(f"Output written to {Path(cache_dir) / 'output.txt'}")